# Experiments

Ноутбук вызывает уже вынесенный пакетный код `mcips` (логика не дублируется здесь).

Черновой исходный ноутбук с сырым кодом - `mcips_prototype.ipynb`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from mcips import Problem, inner_box_search, parallelepiped_search
from mcips.geometry import rotation_2d

## Задача: тонкая наклонная полоса

In [ ]:
problem = Problem(
    c=[1.0, 1.0],
    A=[[1, -1], [-1, 1], [1, 1]],
    b=[1.0, 0.0, 10.0],
    l_bounds=[0, 0],
    u_bounds=[6, 6],
    int_idx=[0],
)
basis = rotation_2d(np.radians(45))

for tau in [0.5, 1.0, 1.5, 2.0]:
    box = inner_box_search(problem, tau=tau)
    par = parallelepiped_search(problem, basis, tau=tau)
    bo = f"{box['center_objective']:.3f}" if box['status'] == 'ok' else 'INFEAS'
    po = f"{par['center_objective']:.3f}" if par['status'] == 'ok' else 'INFEAS'
    print(f"tau={tau}: cube={bo}  parallelepiped={po}")

## Sweep по tau

In [ ]:
taus = np.linspace(0.2, 2.6, 25)
box_obj, par_obj = [], []
for tau in taus:
    b = inner_box_search(problem, tau=tau)
    p = parallelepiped_search(problem, basis, tau=tau)
    box_obj.append(b['center_objective'] if b['status'] == 'ok' else np.nan)
    par_obj.append(p['center_objective'] if p['status'] == 'ok' else np.nan)

plt.figure(figsize=(6, 4))
plt.plot(taus, box_obj, '-o', ms=3, label='cube')
plt.plot(taus, par_obj, '-o', ms=3, label='parallelepiped')
plt.xlabel('tau'); plt.ylabel('c.center'); plt.legend(); plt.grid(alpha=0.3)
plt.title('objective vs tau'); plt.show()

## Sweep по углу базиса

Какой поворот базиса даёт лучший objective при фиксированном tau.

In [ ]:
tau = 1.5
degs = np.arange(0, 90, 3)
objs = []
for d in degs:
    r = parallelepiped_search(problem, rotation_2d(np.radians(d)), tau=tau)
    objs.append(r['center_objective'] if r['status'] == 'ok' else np.nan)

best = degs[int(np.nanargmax(objs))]
print('best angle deg =', best)
plt.figure(figsize=(6, 4))
plt.plot(degs, objs, '-o', ms=3)
plt.xlabel('angle, deg'); plt.ylabel('c.center'); plt.grid(alpha=0.3)
plt.title(f'objective vs basis angle (tau={tau})'); plt.show()